In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.getOrCreate()

In [4]:
data = spark.read.csv(
    "E:/Cluster Computing/Dataset/ca3_customer_clustering_sample.csv",
    header = True,
    inferSchema = True
)

In [5]:
data.show()

+----------+---+------------+-------------+
|CustomerID|Age|AnnualIncome|SpendingScore|
+----------+---+------------+-------------+
|       101| 22|          25|           80|
|       102| 25|          30|           75|
|       103| 23|          28|           85|
|       104| 45|          80|           20|
|       105| 48|          85|           25|
|       106| 42|          78|           18|
|       107| 31|          50|           50|
|       108| 35|          55|           55|
|       109| 33|          48|           48|
|       110| 60|          20|           30|
|       111| 58|          22|           25|
|       112| 62|          18|           35|
+----------+---+------------+-------------+



In [6]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols = ["Age", "AnnualIncome", "SpendingScore"],
    outputCol = "feature"
)
data = assembler.transform(data)

In [7]:
data.show()

+----------+---+------------+-------------+----------------+
|CustomerID|Age|AnnualIncome|SpendingScore|         feature|
+----------+---+------------+-------------+----------------+
|       101| 22|          25|           80|[22.0,25.0,80.0]|
|       102| 25|          30|           75|[25.0,30.0,75.0]|
|       103| 23|          28|           85|[23.0,28.0,85.0]|
|       104| 45|          80|           20|[45.0,80.0,20.0]|
|       105| 48|          85|           25|[48.0,85.0,25.0]|
|       106| 42|          78|           18|[42.0,78.0,18.0]|
|       107| 31|          50|           50|[31.0,50.0,50.0]|
|       108| 35|          55|           55|[35.0,55.0,55.0]|
|       109| 33|          48|           48|[33.0,48.0,48.0]|
|       110| 60|          20|           30|[60.0,20.0,30.0]|
|       111| 58|          22|           25|[58.0,22.0,25.0]|
|       112| 62|          18|           35|[62.0,18.0,35.0]|
+----------+---+------------+-------------+----------------+



In [8]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler(
    inputCol = "feature",
    outputCol = "scaledFeature"
)
sclaer_data = scaler.fit(data)
data = sclaer_data.transform(data)

In [9]:
data.show()

+----------+---+------------+-------------+----------------+--------------------+
|CustomerID|Age|AnnualIncome|SpendingScore|         feature|       scaledFeature|
+----------+---+------------+-------------+----------------+--------------------+
|       101| 22|          25|           80|[22.0,25.0,80.0]|[1.52387226570177...|
|       102| 25|          30|           75|[25.0,30.0,75.0]|[1.73167302920655...|
|       103| 23|          28|           85|[23.0,28.0,85.0]|[1.59313918687003...|
|       104| 45|          80|           20|[45.0,80.0,20.0]|[3.11701145257180...|
|       105| 48|          85|           25|[48.0,85.0,25.0]|[3.32481221607659...|
|       106| 42|          78|           18|[42.0,78.0,18.0]|[2.90921068906701...|
|       107| 31|          50|           50|[31.0,50.0,50.0]|[2.14727455621613...|
|       108| 35|          55|           55|[35.0,55.0,55.0]|[2.42434224088918...|
|       109| 33|          48|           48|[33.0,48.0,48.0]|[2.28580839855265...|
|       110| 60|

In [10]:
from pyspark.ml.clustering import KMeans
kmeans = KMeans(
    k = 3,
    featuresCol = "scaledFeature",
    predictionCol = "cluster"
)

In [11]:
model = kmeans.fit(data)

In [12]:
predictions = model.transform(data)

In [14]:
predictions.show()

+----------+---+------------+-------------+----------------+--------------------+-------+
|CustomerID|Age|AnnualIncome|SpendingScore|         feature|       scaledFeature|cluster|
+----------+---+------------+-------------+----------------+--------------------+-------+
|       101| 22|          25|           80|[22.0,25.0,80.0]|[1.52387226570177...|      1|
|       102| 25|          30|           75|[25.0,30.0,75.0]|[1.73167302920655...|      1|
|       103| 23|          28|           85|[23.0,28.0,85.0]|[1.59313918687003...|      1|
|       104| 45|          80|           20|[45.0,80.0,20.0]|[3.11701145257180...|      2|
|       105| 48|          85|           25|[48.0,85.0,25.0]|[3.32481221607659...|      2|
|       106| 42|          78|           18|[42.0,78.0,18.0]|[2.90921068906701...|      2|
|       107| 31|          50|           50|[31.0,50.0,50.0]|[2.14727455621613...|      1|
|       108| 35|          55|           55|[35.0,55.0,55.0]|[2.42434224088918...|      1|
|       10

In [15]:
from pyspark.ml.evaluation import ClusteringEvaluator

In [18]:
evaluator = ClusteringEvaluator(
    featuresCol = "scaledFeature",
    predictionCol = "cluster",
    metricName = "silhouette"
)

silhouette = evaluator.evaluate(predictions)
silhouette

0.8323903133062901